In [ ]:
# =============================================================
# 清洗从当当网获取销量排名前 50 的 Python 类书籍信息
# 杨金梅25210277 马琳琳25210209
# =============================================================

import pandas as pd
import re
import os

# ===================== 路径配置（已修改为上级目录相对路径） =====================
RAW_DATA_PATH = os.path.join("..", "data_raw", "dangdang_python_books_raw.csv")
CLEAN_DATA_PATH = os.path.join("..", "data_clean", "dangdang_python_books_clean.csv")
DOC_PATH = os.path.join("..", "data_clean", "数据说明文档.md")

os.makedirs(os.path.dirname(CLEAN_DATA_PATH), exist_ok=True)

# ===================== 工具函数 =====================
def remove_currency_symbol(price_str):
    if pd.isna(price_str) or price_str == "":
        return price_str
    return re.sub(r"¥", "", str(price_str)).strip()

def convert_discount_to_percent(discount_str):
    # ✅ 修复：把错误的 price_str 改成 discount_str
    if pd.isna(discount_str) or discount_str == "":
        return discount_str, discount_str
    
    discount_clean = re.sub(r"[折|折扣|无]", "", str(discount_str)).strip()
    try:
        discount_num = float(discount_clean)
        discount_percent = f"{discount_num * 10:.0f}%"
        return discount_clean, discount_percent
    except:
        return "", ""

def clean_publisher(publisher_str):
    if pd.isna(publisher_str) or publisher_str == "":
        return publisher_str
    publisher_clean = re.split(r"出版社", str(publisher_str))[0] + "出版社"
    return publisher_clean.strip()

def split_author_name_country(author_str):
    """
    ✅ 终极兼容所有括号：
    半角：( ) [ ]
    全角：（）【】
    例如：（美）、(美)、【英】、[德] 全部正常拆分
    """
    if pd.isna(author_str) or author_str == "":
        return author_str, author_str
    
    author_str = str(author_str).strip()

    # 匹配所有中英文括号
    country_match = re.search(r"[\[\(【（](.*?)[\]\)】）]", author_str)
    
    author_country = country_match.group(1).strip() if country_match else ""
    author_name = re.sub(r"[\[\(【（].*?[\]\)】）]", "", author_str).strip() or ""
    
    return author_name, author_country

# ===================== 【条件1 增强】50条有效数据 + 无缺失/无重复/无异常值 =====================
def data_quality_check(df):
    # 1. 去重
    df = df.drop_duplicates(subset=["书名"], keep="first")
    
    # 2. 七大核心字段强制无缺失
    core_cols = ["书名", "作者", "出版年份", "出版社", "原价", "折后价", "评论数"]
    df = df.dropna(subset=core_cols)
    df = df[df[core_cols].ne("").all(axis=1)]
    
    # 3. 价格异常值过滤
    df = df[(df["原价"].replace("", "0").astype(float) > 0) &
            (df["折后价"].replace("", "0").astype(float) > 0)]
    
    # 4. 保留前50条有效数据
    df = df.head(50).reset_index(drop=True)
    df["销量排名"] = range(1, len(df)+1)
    return df

# ===================== 【条件2 增强】格式标准化 =====================
def format_standardize(df):
    # 价格 → 浮点型
    df["原价"] = pd.to_numeric(df["原价"], errors="coerce").fillna(0.0).astype(float)
    df["折后价"] = pd.to_numeric(df["折后价"], errors="coerce").fillna(0.0).astype(float)
    
    # 评论数 → 整型
    df["评论数"] = pd.to_numeric(df["评论数"], errors="coerce").fillna(0).astype(int)
    
    # 出版年份 → 字符型
    df["出版年份"] = df["出版年份"].astype(str).str.strip()
    
    # 其余字段 → 规范字符型
    str_cols = ["书名", "作者", "作者姓名", "作者国家", "出版社", "折扣", "百分比折扣", "简介"]
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace("nan", "")
    return df

# ===================== 【条件3 增强】自动生成数据说明文档 =====================
def generate_data_document():
    doc_content = """# 当当网Python书籍数据说明文档
## 一、数据集基本信息
- 数据来源：当当网图书频道
- 数据量级：50条有效数据
- 数据用途：图书价格、评论、出版信息分析

## 二、字段说明与格式
1. 销量排名：int → 销量排序
2. 书名：str → 书籍全称
3. 作者：str → 完整作者名
4. 作者姓名：str → 去除国籍后的作者名
5. 作者国家：str → 作者国籍
6. 出版年份：str → 出版年份（4位数字）
7. 出版社：str → 出版社名称
8. 原价：float → 书籍定价（元）
9. 折后价：float → 实际售价（元）
10. 折扣：str → 折扣（如：8.5）
11. 百分比折扣：str → 百分比折扣（如：85%）
12. 评论数：int → 评论总量
13. 简介：str → 书籍简介

## 三、清洗与标准化流程
1. 去除价格货币符号（¥）
2. 拆分作者姓名与国籍
3. 清洗出版社名称
4. 折扣格式标准化
5. 去重、去缺失、去异常值
6. 保留50条有效数据
7. 字段类型强制标准化
8. 生成规范CSV文件
"""
    with open(DOC_PATH, "w", encoding="utf-8") as f:
        f.write(doc_content)
    print(f"📄 数据说明文档已生成：{DOC_PATH}")

# ===================== 核心清洗逻辑 =====================
def clean_data():
    print("📊 读取原始数据...")
    df = pd.read_csv(RAW_DATA_PATH, encoding="utf-8-sig")
    print(f"原始数据形状：{df.shape}")

    df.loc[df["书名"].str.contains("Python网络爬虫项目开发全程实录", na=False), "作者"] = "明日科技"
    
    print("🔧 清洗价格字段...")
    df["原价"] = df["原价"].apply(remove_currency_symbol)
    df["折后价"] = df["折后价"].apply(remove_currency_symbol)
    
    print("🔧 清洗折扣字段...")
    discount_result = df["折扣"].apply(convert_discount_to_percent)
    df["折扣"] = [x[0] for x in discount_result]
    df["百分比折扣"] = [x[1] for x in discount_result]
    
    print("🔧 清洗出版社字段...")
    df["出版社"] = df["出版社"].apply(clean_publisher)
    
    print("🔧 拆分作者姓名和国家...")
    author_result = df["作者"].apply(split_author_name_country)
    df["作者姓名"] = [x[0] for x in author_result]
    df["作者国家"] = [x[1] for x in author_result]
    
    # ===================== 【新增：条件1 质量校验】 =====================
    print("✅ 数据质量校验：去重、去缺失、去异常、保留50条有效数据")
    df = data_quality_check(df)
    
    # ===================== 【新增：条件2 格式标准化】 =====================
    print("✅ 格式标准化：价格浮点/评论数整型/年份字符")
    df = format_standardize(df)
    
    final_columns = [
        "销量排名", "书名", "作者",
        "作者姓名", "作者国家",
        "出版年份", "出版社",
        "原价", "折后价",
        "折扣", "百分比折扣",
        "评论数", "简介"
    ]
    df_final = df[final_columns].copy()
    
    df_final.to_csv(CLEAN_DATA_PATH, index=False, encoding="utf-8-sig")

    # ===================== 【新增：条件3 生成文档】 =====================
    generate_data_document()
    
    print("\n===== 数据清洗完成！=====")
    print(f"💾 已保存至：{CLEAN_DATA_PATH}")
    print(f"✅ 最终有效数据：{len(df_final)} 条")
    return df_final

# ===================== 主程序执行 =====================
if __name__ == "__main__":
    clean_data()

📊 读取原始数据...
原始数据形状：(50, 10)
🔧 清洗价格字段...
🔧 清洗折扣字段...
🔧 清洗出版社字段...
🔧 拆分作者姓名和国家...
✅ 数据质量校验：去重、去缺失、去异常、保留50条有效数据
✅ 格式标准化：价格浮点/评论数整型/年份字符
📄 数据说明文档已生成：..\data_clean\数据说明文档.md

===== 数据清洗完成！=====
💾 已保存至：..\data_clean\dangdang_python_books_clean.csv
✅ 最终有效数据：50 条
